# Entrenamiento con datasets reales de papa

Clasificación de tres clases: `tizon_tardio`, `tizon_temprano`, `hoja_sana`.
1. PlantVillage: **clasificar la imagen** de una hoja.
2. PlantDoc: **localizar y clasificar hojas completas**; sus cajas no son lesiones. Solo se incluyen las dos enfermedades, sin hoja sana.

El pipeline de preparación conserva fuentes, revisiones y manifiestos. Los conjuntos
son pilotos públicos y necesitan revisión manual y evaluación local independiente.
No se entrena ni se evalúa con las imágenes dibujadas de `data/dataset_yolo`.
No se incorpora todavía minadora, pulgón ni polilla.

Las celdas de entrenamiento consumen recursos: usa GPU en Colab si está disponible.
Estos pesos no sustituyen automáticamente el modelo de seis clases del servidor.


In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['IOTTESIS_ROOT'] = '/content/drive/MyDrive/Tesis_IOT_Papa'

In [9]:
from pathlib import Path

proyecto = Path(os.environ['IOTTESIS_ROOT'])

print("Carpeta del proyecto:", proyecto.exists())
print("Preparador:", (proyecto / "scripts/preparar_datasets_reales.py").exists())
%pip install -q "ultralytics>=8.3,<9" Pillow PyYAML


In [10]:
import os
from pathlib import Path

os.environ['IOTTESIS_ROOT'] = '/content/drive/MyDrive/Tesis_IOT_Papa'
raiz = Path(os.environ['IOTTESIS_ROOT'])

print("Carpeta encontrada:", raiz.is_dir())
print("Script encontrado:",
      (raiz / 'scripts/preparar_datasets_reales.py').is_file())

In [5]:
%pip install -q Pillow PyYAML

In [13]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        str(raiz / 'scripts/preparar_datasets_reales.py'),
        '--root', str(raiz),
        '--source', 'all'
    ],
    check=True
)

In [ ]:
import json

for nombre in [
    'papa_plantvillage_clasificacion',
    'papa_plantdoc_yolo'
]:
    archivo = raiz / 'data/processed' / nombre / 'report.json'

    if archivo.exists():
        reporte = json.loads(archivo.read_text())
        print(nombre, '→', reporte['images'], 'imágenes')
    else:
        print(nombre, '→ preparación pendiente')

In [ ]:
%pip install -q "ultralytics>=8.3,<9"

In [ ]:
import torch

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import sys
import torch
import subprocess

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("GPU disponible:", torch.cuda.is_available())

try:
    resultado = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    )
    print(resultado.stdout or resultado.stderr)
except FileNotFoundError:
    print("nvidia-smi no está disponible en este servidor.")

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import yaml
import json

candidatas = ([Path(os.environ['IOTTESIS_ROOT'])] if os.environ.get('IOTTESIS_ROOT') else [])
candidatas += [Path.cwd(), *Path.cwd().parents, Path('/content/IOTtesis')]
PROJECT_ROOT = next((p.resolve() for p in candidatas
                     if (p / 'scripts/preparar_datasets_reales.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Clona/sube el proyecto y configura IOTTESIS_ROOT con su carpeta raíz.')
subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts/preparar_datasets_reales.py'),
                '--root', str(PROJECT_ROOT)], check=True)
CLASIFICACION = PROJECT_ROOT / 'data/processed/papa_plantvillage_clasificacion'
DETECCION = PROJECT_ROOT / 'data/processed/papa_plantdoc_yolo'
RESULTADOS = PROJECT_ROOT / 'runs/datasets_reales'
for carpeta in [CLASIFICACION, DETECCION]:
    reporte = json.loads((carpeta / 'report.json').read_text())
    print(reporte['source'], reporte['images'], reporte['images_per_class'])
# La configuración de detección debe apuntar a la ubicación actual.
config = yaml.safe_load((DETECCION / 'data.yaml').read_text())
config['path'] = str(DETECCION)
(DETECCION / 'data.yaml').write_text(yaml.safe_dump(config, sort_keys=False))


## A. Clasificación de hojas con PlantVillage

La carpeta por clase es la etiqueta; no se inventan cajas. El test es interno a
PlantVillage y no representa todavía condiciones reales del invernadero. La clase
sana tiene menos muestras: revisar métricas por clase, además de accuracy global.


In [ ]:
from ultralytics import YOLO
import torch

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
clasificador = YOLO('yolo11n-cls.pt')
resultado_clasificacion = clasificador.train(
    data=str(CLASIFICACION), epochs=50, imgsz=224, batch=16,
    seed=42, deterministic=True, device=DEVICE, workers=2,
    project=str(RESULTADOS), name='clasificacion_plantvillage')


In [ ]:
pesos_clasificacion = Path(clasificador.trainer.save_dir) / 'weights/best.pt'
clasificador_final = YOLO(str(pesos_clasificacion))
metricas_clasificacion = clasificador_final.val(data=str(CLASIFICACION), split='test', device=DEVICE)
print('Clases (orden aprendido, no asumir IDs del detector):', clasificador_final.names)
print(metricas_clasificacion.results_dict)


## B. Detección de hojas completas con PlantDoc

Entrenamiento independiente. Este piloto redistribuye los grupos del TRAIN y TEST
original para preparar un piloto con las dos enfermedades. No es comparable como
benchmark oficial PlantDoc. La categoría genérica `Potato leaf` se excluye: no confirma hoja sana.
Las etiquetas de enfermedad también necesitan revisión agronómica. La superficie de las cajas no mide daño o severidad de enfermedad.


In [ ]:
detector = YOLO('yolo11n.pt')
resultado_deteccion = detector.train(
    data=str(DETECCION / 'data.yaml'), epochs=50, imgsz=640, batch=8,
    seed=42, deterministic=True, device=DEVICE, workers=2,
    project=str(RESULTADOS), name='deteccion_hojas_plantdoc')


In [ ]:
pesos_deteccion = Path(detector.trainer.save_dir) / 'weights/best.pt'
detector_final = YOLO(str(pesos_deteccion))
metricas_deteccion = detector_final.val(data=str(DETECCION / 'data.yaml'), split='test', device=DEVICE)
print(detector_final.names)
print(metricas_deteccion.results_dict)


## Registro de resultados

Conserva `args.yaml`, `results.csv`, matrices de confusión y `best.pt` de cada ejecución.
Las métricas anteriores se calculan sobre los datos; no se insertan valores de ejemplo.
Usa val para seleccionar configuraciones y reserva test para la evaluación final.
Antes del despliegue: revisión agronómica de clases, test local sin imágenes repetidas,
y adaptación explícita del backend a clasificación o detección de hojas.
